# Limpieza y validación de datos

## Objetivo

Preparar los datos de transacciones de la tienda online para el análisis comercial, aplicando reglas de limpieza justificadas y verificando la calidad del resultado.

## 1. Reglas de limpieza

Las decisiones de limpieza se basan en los problemas identificados durante el análisis exploratorio.

| Problema detectado | Regla de tratamiento |
|---|---|
| Valores ausentes en `CustomerID` | Conservar los registros. Se excluirán de los análisis que requieran identificar clientes. |
| Valores ausentes en `Description` | Conservarlos, sin inventar ni completar descripciones. |
| Filas duplicadas exactas | Eliminar las filas completamente duplicadas, después de comprobar su impacto. |
| Facturas canceladas (`InvoiceNo` comienza por `C`) | Conservar los registros para poder analizar las cancelaciones por separado. |
| Cantidades negativas | Conservarlas como posibles devoluciones o ajustes, sin asumir que todas tienen el mismo significado. |
| Precios unitarios iguales a cero o negativos | Conservarlos inicialmente y tratarlos de forma diferenciada en el análisis. |
| Valores extremos en cantidades o precios | No eliminarlos automáticamente; solo se modificarán si se identifica un error justificable. |
| Códigos de producto especiales | Conservarlos y distinguirlos de los productos ordinarios cuando sea necesario. |

## 2. Lectura y copia del dataset

In [2]:
import pandas as pd

df = pd.read_excel("../data/raw/Online_Retail.xlsx")

df_clean = df.copy()

print("Dimensiones del dataset:", df_clean.shape)
display(df_clean.head())

Dimensiones del dataset: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


## 3. Eliminación de duplicados exactos

Durante el análisis exploratorio se identificaron 5.268 filas duplicadas exactas.

Se eliminarán únicamente las filas cuyos valores coincidan en todas las columnas. Se conservará la primera aparición de cada registro.

Esta decisión evita mantener repeticiones idénticas en el conjunto de datos, aunque no implica que se eliminen otras líneas que compartan factura o producto si presentan alguna diferencia.

In [3]:
filas_antes = len(df_clean)

df_clean = df_clean.drop_duplicates().copy()

filas_despues = len(df_clean)

print("Filas antes:", filas_antes)
print("Filas después:", filas_despues)
print("Filas eliminadas:", filas_antes - filas_despues)
print("Duplicados restantes:", df_clean.duplicated().sum())

Filas antes: 541909
Filas después: 536641
Filas eliminadas: 5268
Duplicados restantes: 0


## 4. Tratamiento de valores ausentes

El análisis exploratorio identificó valores ausentes en `CustomerID` y `Description`.

Se conservarán estos registros porque la ausencia de un identificador de cliente o de una descripción no implica que el resto de la transacción sea inválido. En particular, eliminar las filas sin `CustomerID` podría excluir operaciones relevantes del análisis comercial.

En los análisis posteriores, las métricas que requieran identificar clientes se calcularán únicamente con los registros que dispongan de ese identificador.

In [4]:
valores_ausentes = (
    df_clean.isna()
    .sum()
    .sort_values(ascending=False)
)

display(
    valores_ausentes[valores_ausentes > 0]
    .to_frame("Valores ausentes")
)

,Valores ausentes
CustomerID,135037
Description,1454


## 5. Validación de tipos de datos

Antes de realizar transformaciones adicionales, se comprobarán los tipos de datos de las columnas.

Es importante que `InvoiceDate` tenga un tipo de fecha válido y que las columnas numéricas puedan utilizarse en cálculos. Los identificadores de factura y producto deben conservarse sin perder información ni alterar sus valores.

En este paso solo se inspeccionarán los tipos actuales. Cualquier conversión se realizará después de comprobar si es necesaria.

In [5]:
# Comprobar los tipos de datos actuales
display(
    df_clean.dtypes.to_frame("Tipo de dato")
)

,Tipo de dato
InvoiceNo,object
StockCode,object
Description,object
Quantity,int64
InvoiceDate,datetime64[us]
UnitPrice,float64
CustomerID,float64
Country,str


## 6. Conversión del identificador de cliente

La columna `CustomerID` se encuentra en formato `float64` debido a la presencia de valores ausentes. Sin embargo, sus valores representan identificadores de clientes, no cantidades decimales.

Se convertirá al tipo entero nullable `Int64` de pandas. Este tipo permite conservar los identificadores como enteros y mantener los valores ausentes sin eliminarlos ni imputarlos.

In [6]:
df_clean["CustomerID"] = df_clean["CustomerID"].astype("Int64")

print("Tipo de CustomerID:", df_clean["CustomerID"].dtype)
print("Valores ausentes:", df_clean["CustomerID"].isna().sum())

display(df_clean[["CustomerID"]].head(10))

Tipo de CustomerID: Int64
Valores ausentes: 135037


,CustomerID
0,17850
1,17850
2,17850
3,17850
4,17850
5,17850
6,17850
7,17850
8,17850
9,13047


## 7. Identificación de facturas canceladas

Durante el análisis exploratorio se identificaron facturas cuyo número comienza por `C`, asociadas a cancelaciones.

Para facilitar su identificación en los análisis posteriores, se creará la columna booleana `IsCancelled`. Será `True` cuando `InvoiceNo` comience por `C` y `False` en caso contrario.

Los registros se conservarán para permitir analizar las cancelaciones por separado. Esta regla identifica las facturas según su número, pero no implica que todas las cantidades negativas correspondan a cancelaciones.

In [7]:
df_clean["IsCancelled"] = (
    df_clean["InvoiceNo"]
    .astype("string")
    .str.startswith("C", na=False)
)

print("Facturas canceladas (filas):", df_clean["IsCancelled"].sum())
print("Filas no identificadas como canceladas:", (~df_clean["IsCancelled"]).sum())

display(
    df_clean.loc[
        df_clean["IsCancelled"],
        ["InvoiceNo", "Quantity", "UnitPrice", "IsCancelled"]
    ].head()
)

Facturas canceladas (filas): 9251
Filas no identificadas como canceladas: 527390


,InvoiceNo,Quantity,UnitPrice,IsCancelled
141,C536379,-1,27.50,True
154,C536383,-1,4.65,True
235,C536391,-12,1.65,True
236,C536391,-24,0.29,True
237,C536391,-24,0.29,True


## 8. Identificación de cantidades y precios especiales

El análisis exploratorio detectó cantidades negativas, precios unitarios iguales a cero y precios unitarios negativos.

Estos valores no se eliminarán automáticamente, ya que pueden corresponder a devoluciones, ajustes contables u otras operaciones comerciales.

Se crearán indicadores para facilitar su identificación y permitir que las reglas de análisis posteriores distingan estos registros de las operaciones ordinarias.

In [8]:
df_clean["IsNegativeQuantity"] = df_clean["Quantity"] < 0

df_clean["IsZeroPrice"] = df_clean["UnitPrice"] == 0

df_clean["IsNegativePrice"] = df_clean["UnitPrice"] < 0

print("Filas con cantidad negativa:", df_clean["IsNegativeQuantity"].sum())
print("Filas con precio igual a cero:", df_clean["IsZeroPrice"].sum())
print("Filas con precio negativo:", df_clean["IsNegativePrice"].sum())

Filas con cantidad negativa: 10587
Filas con precio igual a cero: 2510
Filas con precio negativo: 2


## 9. Cálculo del importe por línea

Se creará la columna `LineTotal`, calculada multiplicando `Quantity` por `UnitPrice`.

Esta columna representa el importe aritmético de cada línea de transacción. Puede contener valores negativos cuando la cantidad o el precio sean negativos.

El importe se conservará sin eliminar operaciones especiales. Su interpretación como indicador de ventas se definirá posteriormente, teniendo en cuenta las cancelaciones y los ajustes identificados durante el análisis exploratorio.

In [9]:
df_clean["LineTotal"] = df_clean["Quantity"] * df_clean["UnitPrice"]

print("Valores ausentes en LineTotal:", df_clean["LineTotal"].isna().sum())
print("Importe mínimo:", df_clean["LineTotal"].min())
print("Importe máximo:", df_clean["LineTotal"].max())

display(
    df_clean[
        ["InvoiceNo", "Quantity", "UnitPrice", "LineTotal"]
    ].head(10)
)

Valores ausentes en LineTotal: 0
Importe mínimo: -168469.6
Importe máximo: 168469.6


,InvoiceNo,Quantity,UnitPrice,LineTotal
0,536365,6,2.55,15.30
1,536365,6,3.39,20.34
2,536365,8,2.75,22.00
3,536365,6,3.39,20.34
4,536365,6,3.39,20.34
5,536365,2,7.65,15.30
6,536365,6,4.25,25.50
7,536366,6,1.85,11.10
8,536366,6,1.85,11.10
9,536367,32,1.69,54.08


## 10. Validación de los datos tras la limpieza

Una vez aplicadas las transformaciones, se realizarán comprobaciones de consistencia para verificar que:

- Se han eliminado los duplicados exactos.
- Se conserva el número esperado de filas.
- Los indicadores de cancelación, cantidades negativas y precios especiales están correctamente definidos.
- `LineTotal` coincide con el producto de `Quantity` por `UnitPrice`.
- Se mantienen los valores ausentes que se han decidido conservar.

In [10]:
print("Filas:", len(df_clean))
print("Columnas:", len(df_clean.columns))
print("Duplicados exactos:", df_clean.duplicated().sum())

display(
    df_clean.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("Valores ausentes")
    .query("`Valores ausentes` > 0")
)

print(
    "Cancelaciones:",
    df_clean["IsCancelled"].sum()
)
print(
    "Cantidades negativas:",
    df_clean["IsNegativeQuantity"].sum()
)
print(
    "Precios iguales a cero:",
    df_clean["IsZeroPrice"].sum()
)
print(
    "Precios negativos:",
    df_clean["IsNegativePrice"].sum()
)

print(
    "\nImportes calculados correctamente:",
    (df_clean["LineTotal"] == df_clean["Quantity"] * df_clean["UnitPrice"]).all()
)

Filas: 536641
Columnas: 13
Duplicados exactos: 0


,Valores ausentes
CustomerID,135037
Description,1454


Cancelaciones: 9251
Cantidades negativas: 10587
Precios iguales a cero: 2510
Precios negativos: 2

Importes calculados correctamente: True


## 11. Guardado de los datos limpios

Se guardará el conjunto de datos resultante en la carpeta `data/processed/`, manteniendo intacto el archivo original de `data/raw/`.

El archivo procesado incluirá las transformaciones realizadas, los indicadores creados y la columna `LineTotal`. Se utilizará el formato CSV para facilitar su posterior lectura desde Python, SQL y Power BI.

In [11]:
path="../data/processed/online_retail_clean.csv"

df_clean.to_csv(path)

print("Archivo guardado en:", path)
print("Filas guardadas:", len(df_clean))
print("Columnas guardadas:", len(df_clean.columns))

Archivo guardado en: ../data/processed/online_retail_clean.csv
Filas guardadas: 536641
Columnas guardadas: 13


## 12. Conclusiones de la limpieza y validación

La fase de limpieza ha permitido preparar un conjunto de datos para el análisis comercial, conservando el archivo original sin modificaciones.

### Transformaciones realizadas

- Eliminación de 5.268 filas duplicadas exactas.
- Conservación de los registros con valores ausentes en `CustomerID` y `Description`.
- Conversión de `CustomerID` al tipo entero nullable `Int64`.
- Creación de `IsCancelled` para identificar facturas cuyo número comienza por `C`.
- Creación de indicadores para cantidades negativas, precios iguales a cero y precios negativos.
- Cálculo de `LineTotal` como producto de `Quantity` por `UnitPrice`.

### Validación del resultado

El conjunto procesado contiene **536.641 filas y 13 columnas**, no presenta duplicados exactos y el cálculo de `LineTotal` se ha validado correctamente.

Se han conservado los registros con cantidades negativas, precios especiales y valores ausentes, evitando eliminaciones automáticas sin justificación.

### Resultado

El archivo `data/processed/online_retail_clean.csv` queda preparado para la siguiente fase del proyecto. Las métricas comerciales deberán definirse teniendo en cuenta las cancelaciones y los movimientos especiales, para evitar interpretar todos los importes como ventas ordinarias.